# mBART Hinglish → English Evaluation Pipeline

**Evaluates the fine-tuned mBART LoRA checkpoint on the held-out test set.**

Metrics computed:
- **Loss** (cross-entropy on tokenized test set)
- **SacreBLEU** (corpus-level BLEU via sacrebleu)
- **chrF** (character n-gram F-score)
- **BERTScore F1** (semantic similarity via contextual embeddings)

> Only `hi_IN → en_XX` pairs are used for BLEU/chrF/BERTScore, matching your training `compute_metrics` logic.

## Cell 1 — Install dependencies

In [ ]:
# Uninstall conflicting versions first
!pip uninstall -y numpy pandas torch torchvision torchaudio transformers datasets peft accelerate bitsandbytes evaluate sacrebleu huggingface_hub triton 2>/dev/null

# Install the exact same stack used during training
!pip install -q \
  numpy==1.26.4 \
  pandas==2.2.2 \
  torch==2.2.2 \
  torchvision==0.17.2 \
  torchaudio==2.2.2 \
  transformers==4.44.2 \
  datasets==2.21.0 \
  peft==0.12.0 \
  accelerate==0.33.0 \
  bitsandbytes==0.46.1 \
  evaluate==0.4.2 \
  sacrebleu==2.4.3 \
  huggingface_hub==0.24.6 \
  sentencepiece \
  bert_score \
  tqdm

print('\n✅ All packages installed successfully.')

## Cell 2 — Restart runtime (REQUIRED after install)

In [ ]:
# ⚠️ Run this cell immediately after the install cell above.
# This forces Colab to reload the newly installed package versions.
import os
os.kill(os.getpid(), 9)

## Cell 3 — Sanity-check environment

In [ ]:
import torch
import transformers

print('Torch       :', torch.__version__)
print('Transformers:', transformers.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

## Cell 4 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive mounted.')

## Cell 5 — Config: paths & hyperparameters

In [ ]:
import os

# ── Paths (mirror your training notebook exactly) ────────────────────────────
OUTPUT_DIR        = '/content/drive/MyDrive/shared/mBART_Finetune_Hinglish/pacman/bidirect'
DRIVE_DATA_PATH   = '/content/drive/MyDrive/shared/mBART_Finetune_Hinglish/pacman/cleaned_data'
TOKENIZED_DATA_DIR = os.path.join(OUTPUT_DIR, 'tokenized_data')
CHECKPOINT_DIR    = os.path.join(OUTPUT_DIR, 'updated_checkpoint')

# ── Base model ───────────────────────────────────────────────────────────────
MODEL_NAME = 'facebook/mbart-large-50-many-to-many-mmt'

# ── Eval settings ────────────────────────────────────────────────────────────
EVAL_BATCH_SIZE    = 8          # Increase if GPU VRAM allows (T4 ~16 GB)
MAX_LENGTH         = 128        # Must match training
NUM_BEAMS          = 5          # Must match training generation_num_beams
TEST_SAMPLE_LIMIT  = None       # Set to e.g. 500 for a quick smoke-test; None = full test set

print('Config loaded:')
print(f'  CHECKPOINT_DIR    : {CHECKPOINT_DIR}')
print(f'  TOKENIZED_DATA_DIR: {TOKENIZED_DATA_DIR}')
print(f'  EVAL_BATCH_SIZE   : {EVAL_BATCH_SIZE}')
print(f'  TEST_SAMPLE_LIMIT : {TEST_SAMPLE_LIMIT}')

## Cell 6 — Find the latest checkpoint

In [ ]:
def find_latest_checkpoint(ckpt_dir):
    """Return the path of the highest-numbered checkpoint-NNNN directory."""
    if not os.path.exists(ckpt_dir):
        raise FileNotFoundError(f'Checkpoint dir not found: {ckpt_dir}')
    checkpoints = [
        os.path.join(ckpt_dir, d)
        for d in os.listdir(ckpt_dir)
        if d.startswith('checkpoint-') and os.path.isdir(os.path.join(ckpt_dir, d))
    ]
    if not checkpoints:
        raise FileNotFoundError(f'No checkpoint-* dirs found in {ckpt_dir}')
    checkpoints.sort(key=lambda p: int(p.split('-')[-1]))
    latest = checkpoints[-1]
    print(f'All checkpoints found : {[os.path.basename(c) for c in checkpoints]}')
    print(f'Using latest          : {latest}')
    return latest

LATEST_CHECKPOINT = find_latest_checkpoint(CHECKPOINT_DIR)

## Cell 7 — Load tokenizer

In [ ]:
from transformers import MBart50TokenizerFast

tokenizer = MBart50TokenizerFast.from_pretrained(MODEL_NAME)
print('✅ Tokenizer loaded.')

## Cell 8 — Load base model in 4-bit and apply LoRA adapter

In [ ]:
from transformers import AutoModelForSeq2SeqLM, BitsAndBytesConfig
from peft import PeftModel
import torch

# 4-bit quantisation config (identical to training)
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=torch.float16
)

print('Loading base model (this may take ~2-3 min)...')
base_model = AutoModelForSeq2SeqLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map='auto'
)
print('Base model loaded.')

# Wrap with the fine-tuned LoRA adapter from the checkpoint
print(f'Loading LoRA adapter from: {LATEST_CHECKPOINT}')
model = PeftModel.from_pretrained(base_model, LATEST_CHECKPOINT)
model.eval()  # switch to eval mode — disables dropout
print('✅ LoRA adapter applied. Model in eval mode.')

# Print parameter summary
total   = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params: {total:,}  |  Trainable (LoRA): {trainable:,}')

## Cell 9 — Load tokenized test dataset from Drive

In [ ]:
from datasets import load_from_disk

print(f'Loading tokenized test split from: {TOKENIZED_DATA_DIR}/test')
test_dataset = load_from_disk(os.path.join(TOKENIZED_DATA_DIR, 'test'))
print(f'Test set: {test_dataset}')

# Optionally limit for a quick smoke-test
if TEST_SAMPLE_LIMIT and TEST_SAMPLE_LIMIT < len(test_dataset):
    test_dataset = test_dataset.select(range(TEST_SAMPLE_LIMIT))
    print(f'⚠️  Subsampled to {TEST_SAMPLE_LIMIT} examples for quick evaluation.')

print(f'\nFinal test split size: {len(test_dataset)} examples')

## Cell 10 — Load raw test split (needed for lang-pair filtering)

In [ ]:
# The raw (pre-tokenized) test split is needed to identify hi_IN->en_XX pairs.
# We reconstruct it exactly as in training:
#   full dataset → bidirectional expansion → 90/5/5 split
#
# If you saved the raw split to disk, load it directly (faster).
# Otherwise we rebuild from the JSONL file.

RAW_TEST_PATH = os.path.join(OUTPUT_DIR, 'raw_splits', 'test')  # optional fast path

if os.path.exists(RAW_TEST_PATH):
    print(f'Loading raw test split from cached disk path: {RAW_TEST_PATH}')
    raw_test_dataset = load_from_disk(RAW_TEST_PATH)
else:
    print('Raw split not cached on disk — rebuilding from JSONL (may take a minute)...')
    import random
    from datasets import load_dataset, Dataset

    TRAIN_FILE = os.path.join(DRIVE_DATA_PATH, 'train', 'Final_6M_CM_SelectedColumns_train.jsonl')
    raw_ds = load_dataset('json', data_files={'data': TRAIN_FILE})['data']

    INITIAL_DATA_LIMIT = 100_000  # must match training
    if INITIAL_DATA_LIMIT and INITIAL_DATA_LIMIT < len(raw_ds):
        raw_ds = raw_ds.select(range(INITIAL_DATA_LIMIT))

    def make_bidirectional(examples_batch, forward_weight=0.7):
        input_texts, target_texts, src_langs, tgt_langs = [], [], [], []
        for i in range(len(examples_batch['cm_text_r'])):
            cm_text_r = examples_batch['cm_text_r'][i]
            emb_text  = examples_batch['emb_text'][i]
            input_texts.append(cm_text_r)
            target_texts.append(emb_text)
            src_langs.append('hi_IN')
            tgt_langs.append('en_XX')
            if random.random() < (1 - forward_weight):
                input_texts.append(emb_text)
                target_texts.append(cm_text_r)
                src_langs.append('en_XX')
                tgt_langs.append('hi_IN')
        return {'input_text': input_texts, 'target_text': target_texts,
                'src_lang': src_langs, 'tgt_lang': tgt_langs}

    random.seed(42)  # reproducibility
    expanded = raw_ds.map(make_bidirectional, fn_kwargs={'forward_weight': 0.7},
                          batched=True, remove_columns=raw_ds.column_names,
                          load_from_cache_file=False)

    split_1 = expanded.train_test_split(test_size=0.1, seed=42)
    split_2 = split_1['test'].train_test_split(test_size=0.5, seed=42)
    raw_test_dataset = split_2['test']

    print('Caching raw test split for future use...')
    os.makedirs(RAW_TEST_PATH, exist_ok=True)
    raw_test_dataset.save_to_disk(RAW_TEST_PATH)

# Sub-sample raw split to match tokenized test set size
if TEST_SAMPLE_LIMIT and TEST_SAMPLE_LIMIT < len(raw_test_dataset):
    raw_test_dataset = raw_test_dataset.select(range(TEST_SAMPLE_LIMIT))

print(f'\nRaw test split size : {len(raw_test_dataset)}')
print(raw_test_dataset)

## Cell 11 — Compute evaluation LOSS

In [ ]:
import torch
from torch.utils.data import DataLoader
from transformers import DataCollatorForSeq2Seq
import numpy as np

data_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)

test_dataloader = DataLoader(
    test_dataset,
    batch_size=EVAL_BATCH_SIZE,
    collate_fn=data_collator,
    shuffle=False
)

device = next(model.parameters()).device
total_loss = 0.0
total_steps = 0

print(f'Computing loss over {len(test_dataset)} test examples ({len(test_dataloader)} batches)...')

model.eval()
with torch.no_grad():
    for step, batch in enumerate(test_dataloader):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels         = batch['labels'].to(device)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )
        total_loss  += outputs.loss.item()
        total_steps += 1

        if (step + 1) % 50 == 0:
            print(f'  [{step+1}/{len(test_dataloader)}] Running avg loss: {total_loss/total_steps:.4f}')

avg_loss = total_loss / total_steps
perplexity = np.exp(avg_loss)

print(f'\n=== LOSS RESULTS ===')
print(f'Average Cross-Entropy Loss : {avg_loss:.4f}')
print(f'Perplexity                 : {perplexity:.4f}')

## Cell 12 — Generate translations for BLEU / chrF / BERTScore

In [ ]:
from tqdm.auto import tqdm
import torch
from torch.utils.data import DataLoader
from transformers import DataCollatorForSeq2Seq

# We need the 'raw_test_dataset' which was processed in cell 'd9192daf'.
# And 'tokenizer', 'MAX_LENGTH', 'model', 'EVAL_BATCH_SIZE', 'NUM_BEAMS', 'device' from earlier cells.

# Filter the raw_test_dataset to get only 'hi_IN' -> 'en_XX' pairs
# This `raw_test_dataset` has been verified to contain correct SRC/REF alignment (from cell d9192daf)
hi_en_raw_eval = raw_test_dataset.filter(
    lambda example: example['src_lang'] == 'hi_IN' and example['tgt_lang'] == 'en_XX'
)

print(f'Filtered raw hi_IN→en_XX pairs for evaluation: {len(hi_en_raw_eval)}')

# Define the tokenization function for the evaluation dataset
def tokenize_eval_data(examples):
    # Set source and target languages for tokenization
    tokenizer.src_lang = 'hi_IN'
    model_inputs = tokenizer(
        examples['input_text'],
        max_length=MAX_LENGTH,
        truncation=True
    )
    tokenizer.tgt_lang = 'en_XX'
    labels = tokenizer(
        examples['target_text'],
        max_length=MAX_LENGTH,
        truncation=True
    )
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

# Tokenize the filtered raw data to create the evaluation dataset
hi_en_tokenized_eval = hi_en_raw_eval.map(tokenize_eval_data, batched=True)

# Prepare all_labels directly from the raw target text for metric calculation
all_labels = [ex['target_text'] for ex in hi_en_raw_eval]

# ── Generation loop ─────────────────────────────────────────────────────────
all_preds  = []

gen_collator = DataCollatorForSeq2Seq(tokenizer, model=model, padding=True)
gen_dataloader = DataLoader(
    hi_en_tokenized_eval, # Use the newly tokenized dataset
    batch_size=EVAL_BATCH_SIZE,
    collate_fn=gen_collator,
    shuffle=False
)

# Set forced BOS token for English output
EN_LANG_ID = tokenizer.lang_code_to_id['en_XX']

print(f'Generating translations ({len(hi_en_tokenized_eval)} pairs, {len(gen_dataloader)} batches)...')

model.eval()
with torch.no_grad():
    for batch in tqdm(gen_dataloader, desc='Generating'):
        input_ids      = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        generated = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            forced_bos_token_id=EN_LANG_ID,
            max_length=MAX_LENGTH,
            num_beams=NUM_BEAMS,
        )

        decoded_preds  = tokenizer.batch_decode(generated, skip_special_tokens=True)
        all_preds.extend(decoded_preds)

print(f'\n✅ Generation complete. Total predictions: {len(all_preds)}')

# Preview 5 examples
print('\n--- Sample predictions (now with guaranteed alignment) ---')
for i in range(min(5, len(all_preds))):
    print(f'  SRC  : {hi_en_raw_eval[i]["input_text"]}')
    print(f'  REF  : {hi_en_raw_eval[i]["target_text"]}') # Use original target_text from aligned raw dataset
    print(f'  PRED : {all_preds[i]}')
    print()

### Correctly Aligned Source and Reference Samples

To confirm the actual alignment within the `hi_en_raw` dataset, let's display a few `input_text` (Source) and `target_text` (Reference) pairs directly.

In [ ]:
# Print correctly aligned SRC and REF from hi_en_raw
print('\n--- Correctly Aligned Samples (from hi_en_raw) ---')
for i in range(min(5, len(hi_en_raw))):
    print(f'  SRC  : {hi_en_raw[i]["input_text"]}')
    print(f'  REF  : {hi_en_raw[i]["target_text"]}')
    print()

## Cell 13 — Compute SacreBLEU & chrF

In [ ]:
import evaluate

bleu_metric = evaluate.load('sacrebleu')
chrf_metric = evaluate.load('chrf')

# sacrebleu expects list-of-lists for references
bleu_result = bleu_metric.compute(
    predictions=all_preds,
    references=[[ref] for ref in all_labels]
)

chrf_result = chrf_metric.compute(
    predictions=all_preds,
    references=[[ref] for ref in all_labels]
)

print('=== BLEU / chrF RESULTS ===')
print(f'SacreBLEU score : {bleu_result["score"]:.4f}')
print(f'chrF score      : {chrf_result["score"]:.4f}')
print(f'\nDetailed BLEU breakdown:')
for k, v in bleu_result.items():
    print(f'  {k}: {v}')

## Cell 14 — Compute BERTScore (semantic similarity)

In [ ]:
# BERTScore uses a pretrained transformer to compute contextual similarity.
# lang='en' uses roberta-large by default. Switch to 'distilbert-base-uncased' for speed.

bertscore_metric = evaluate.load('bertscore')

print('Computing BERTScore (this may take a few minutes on T4)...')

bert_result = bertscore_metric.compute(
    predictions=all_preds,
    references=all_labels,
    lang='en',
    # model_type='distilbert-base-uncased',  # uncomment for 3× speedup
    batch_size=64
)

import numpy as np
bert_p = np.mean(bert_result['precision'])
bert_r = np.mean(bert_result['recall'])
bert_f = np.mean(bert_result['f1'])

print('=== BERTScore RESULTS ===')
print(f'BERTScore Precision : {bert_p:.4f}')
print(f'BERTScore Recall    : {bert_r:.4f}')
print(f'BERTScore F1        : {bert_f:.4f}')

## Cell 15 — Consolidated results summary

In [ ]:
import pandas as pd

results = {
    'Metric'         : ['Cross-Entropy Loss', 'Perplexity', 'SacreBLEU', 'chrF', 'BERTScore-P', 'BERTScore-R', 'BERTScore-F1'],
    'Value'          : [
        round(avg_loss,   4),
        round(perplexity, 4),
        round(bleu_result['score'], 4),
        round(chrf_result['score'], 4),
        round(bert_p, 4),
        round(bert_r, 4),
        round(bert_f, 4),
    ],
    'Notes': [
        'Lower is better',
        'Lower is better',
        'Higher is better (0-100)',
        'Higher is better (0-100)',
        'Semantic similarity (0-1)',
        'Semantic similarity (0-1)',
        'Semantic similarity (0-1)',
    ]
}

df = pd.DataFrame(results)
print('\n' + '='*60)
print('      FINAL EVALUATION SUMMARY — mBART LoRA Fine-tune')
print(f'      Checkpoint: {os.path.basename(LATEST_CHECKPOINT)}')
print(f'      Test pairs (hi→en): {len(all_preds)}')
print('='*60)
print(df.to_string(index=False))
print('='*60)

## Cell 16 — (Optional) Save results to Drive

In [ ]:
RESULTS_DIR = os.path.join(OUTPUT_DIR, 'evaluation_results')
os.makedirs(RESULTS_DIR, exist_ok=True)

checkpoint_tag = os.path.basename(LATEST_CHECKPOINT)  # e.g. checkpoint-4500

# Save summary CSV
summary_path = os.path.join(RESULTS_DIR, f'summary_{checkpoint_tag}.csv')
df.to_csv(summary_path, index=False)
print(f'Summary saved  → {summary_path}')

# Save raw predictions for error analysis
pred_df = pd.DataFrame({
    'source'    : [hi_en_raw[i]['input_text']  for i in range(len(all_preds))],
    'reference' : all_labels,
    'hypothesis': all_preds
})
pred_path = os.path.join(RESULTS_DIR, f'predictions_{checkpoint_tag}.csv')
pred_df.to_csv(pred_path, index=False)
print(f'Predictions saved → {pred_path}')

## Re-displaying Correctly Aligned Source and Reference Samples (After Session Expiry)

This cell re-establishes necessary configurations and loads the raw test dataset to display correctly aligned source (`input_text`) and reference (`target_text`) pairs, addressing the misalignment observed previously.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive remounted.')

In [ ]:
import os
import random
from datasets import load_dataset, Dataset, load_from_disk
import pandas as pd

# --- Re-establish Configuration (from Cell 5) ---
OUTPUT_DIR        = '/content/drive/MyDrive/mBART_Finetune_Hinglish/pacman/bidirect'
DRIVE_DATA_PATH   = '/content/drive/MyDrive/mBART_Finetune_Hinglish/pacman/cleaned_data'
TOKENIZED_DATA_DIR = os.path.join(OUTPUT_DIR, 'tokenized_data')
CHECKPOINT_DIR    = os.path.join(OUTPUT_DIR, 'updated_checkpoint')

MODEL_NAME = 'facebook/mbart-large-50-many-to-many-mmt'

EVAL_BATCH_SIZE    = 8
MAX_LENGTH         = 128
NUM_BEAMS          = 5
TEST_SAMPLE_LIMIT  = None # Set to None to load full dataset

# --- Re-create find_latest_checkpoint function (from Cell 6) ---
def find_latest_checkpoint(ckpt_dir):
    if not os.path.exists(ckpt_dir):
        raise FileNotFoundError(f'Checkpoint dir not found: {ckpt_dir}')
    checkpoints = [
        os.path.join(ckpt_dir, d)
        for d in os.listdir(ckpt_dir)
        if d.startswith('checkpoint-') and os.path.isdir(os.path.join(ckpt_dir, d))
    ]
    if not checkpoints:
        raise FileNotFoundError(f'No checkpoint-* dirs found in {ckpt_dir}')
    checkpoints.sort(key=lambda p: int(p.split('-')[-1]))
    latest = checkpoints[-1]
    print(f'All checkpoints found : {[os.path.basename(c) for c in checkpoints]}')
    print(f'Using latest          : {latest}')
    return latest

# LATEST_CHECKPOINT is not strictly needed for this task, but included for completeness if other cells depend on it.
try:
    LATEST_CHECKPOINT = find_latest_checkpoint(CHECKPOINT_DIR)
except FileNotFoundError as e:
    print(f"Warning: {e}. Some functionalities might be limited if checkpoints are required.")
    LATEST_CHECKPOINT = None


# --- Re-load raw test dataset (from Cell 10) ---
RAW_TEST_PATH = os.path.join(OUTPUT_DIR, 'raw_splits', 'test')

if os.path.exists(RAW_TEST_PATH):
    print(f'Loading raw test split from cached disk path: {RAW_TEST_PATH}')
    raw_test_dataset = load_from_disk(RAW_TEST_PATH)
else:
    print('Raw split not cached on disk — rebuilding from JSONL (may take a minute)...')

    TRAIN_FILE = os.path.join(DRIVE_DATA_PATH, 'train', 'Final_6M_CM_SelectedColumns_train.jsonl')

    # Step-by-step path verification
    print(f"Checking existence of DRIVE_DATA_PATH: {DRIVE_DATA_PATH} -- {os.path.exists(DRIVE_DATA_PATH)}")
    print(f"Checking existence of {os.path.join(DRIVE_DATA_PATH, 'train')} -- {os.path.exists(os.path.join(DRIVE_DATA_PATH, 'train'))}")
    print(f"Checking existence of TRAIN_FILE: {TRAIN_FILE} -- {os.path.exists(TRAIN_FILE)}")

    raw_ds = load_dataset('json', data_files={'data': TRAIN_FILE})['data']

    INITIAL_DATA_LIMIT = 100_000
    if INITIAL_DATA_LIMIT and INITIAL_DATA_LIMIT < len(raw_ds):
        raw_ds = raw_ds.select(range(INITIAL_DATA_LIMIT))

    def make_bidirectional(examples_batch, forward_weight=0.7):
        input_texts, target_texts, src_langs, tgt_langs = [], [], [], []
        for i in range(len(examples_batch['cm_text_r'])):
            cm_text_r = examples_batch['cm_text_r'][i]
            emb_text  = examples_batch['emb_text'][i]
            input_texts.append(cm_text_r)
            target_texts.append(emb_text)
            src_langs.append('hi_IN')
            tgt_langs.append('en_XX')
            if random.random() < (1 - forward_weight):
                input_texts.append(emb_text)
                target_texts.append(cm_text_r)
                src_langs.append('en_XX')
                tgt_langs.append('hi_IN')
        return {'input_text': input_texts, 'target_text': target_texts,
                'src_lang': src_langs, 'tgt_lang': tgt_langs}

    random.seed(42)
    expanded = raw_ds.map(make_bidirectional, fn_kwargs={'forward_weight': 0.7},
                          batched=True, remove_columns=raw_ds.column_names,
                          load_from_cache_file=False)

    split_1 = expanded.train_test_split(test_size=0.1, seed=42)
    split_2 = split_1['test'].train_test_split(test_size=0.5, seed=42)
    raw_test_dataset = split_2['test']

    print('Caching raw test split for future use...')
    os.makedirs(RAW_TEST_PATH, exist_ok=True)
    raw_test_dataset.save_to_disk(RAW_TEST_PATH)

# Sub-sample raw split to match tokenized test set size (if TEST_SAMPLE_LIMIT was set and applicable)
if TEST_SAMPLE_LIMIT and TEST_SAMPLE_LIMIT < len(raw_test_dataset):
    raw_test_dataset = raw_test_dataset.select(range(TEST_SAMPLE_LIMIT))

print(f'\nRaw test split size : {len(raw_test_dataset)}')
print(raw_test_dataset)

# --- Display correctly aligned SRC and REF (as requested) ---
print('\n--- Correctly Aligned Samples (from hi_en_raw) ---')
# Filter for hi_IN -> en_XX pairs to match the evaluation context
hi_en_raw_filtered = raw_test_dataset.filter(lambda example: example['src_lang'] == 'hi_IN' and example['tgt_lang'] == 'en_XX')

for i in range(min(10, len(hi_en_raw_filtered))):
    print(f'  SRC  : {hi_en_raw_filtered[i]["input_text"]}')
    print(f'  REF  : {hi_en_raw_filtered[i]["target_text"]}')
    print()

In [ ]:
import os
print(os.listdir('/content/drive/MyDrive/'))